# LifePet - VetBot

Assistente veterinário inteligente do LifePet, desenvolvido para o Challenge FIAP 2026 com a Clyvo VET.

O VetBot combina três tecnologias de IA:
- **Gemini 3.5 Flash** como modelo de linguagem
- **Function Calling** para consultar dados reais dos pets
- **RAG** (Retrieval Augmented Generation) para buscar conhecimento veterinário relevante

In [ ]:
!pip install -q -U "google-genai>=2.3.0" "pydantic>=2.0" "numpy>=1.26"

## 1. Configuração
Instalação das bibliotecas e conexão com a API do Gemini.

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash"

print("Cliente criado com sucesso.")

## 2. Base de dados dos pets
Perfis simulados de pets com histórico de vacinas, consultas e medicamentos.

In [ ]:
import json

PETS = [
    {
        "id": 1,
        "nome": "Thor",
        "especie": "cachorro",
        "raca": "Golden Retriever",
        "idade": 3,
        "peso_kg": 28.5,
        "tutor": "Carlos Silva",
        "vacinas": [
            {"nome": "V10", "data": "2024-03-10", "proxima": "2025-03-10"},
            {"nome": "Antirrábica", "data": "2024-03-10", "proxima": "2025-03-10"},
        ],
        "consultas": [
            {"data": "2024-06-15", "motivo": "check-up anual", "veterinario": "Dr. Souza"},
        ],
        "medicamentos": [],
        "observacoes": "Alérgico a frango. Ativo e saudável."
    },
    {
        "id": 2,
        "nome": "Luna",
        "especie": "gato",
        "raca": "Siamês",
        "idade": 5,
        "peso_kg": 4.2,
        "tutor": "Ana Oliveira",
        "vacinas": [
            {"nome": "V4", "data": "2023-11-20", "proxima": "2024-11-20"},
        ],
        "consultas": [
            {"data": "2024-01-10", "motivo": "vomitos frequentes", "veterinario": "Dra. Lima"},
        ],
        "medicamentos": [
            {"nome": "Omeprazol", "dose": "1 comprimido", "frequencia": "diaria", "ate": "2024-02-10"}
        ],
        "observacoes": "Histórico de sensibilidade gastrointestinal."
    },
    {
        "id": 3,
        "nome": "Bob",
        "especie": "cachorro",
        "raca": "Bulldog Francês",
        "idade": 7,
        "peso_kg": 12.0,
        "tutor": "Pedro Mendes",
        "vacinas": [
            {"nome": "V10", "data": "2024-01-05", "proxima": "2025-01-05"},
        ],
        "consultas": [
            {"data": "2024-08-20", "motivo": "dificuldade respiratória", "veterinario": "Dr. Costa"},
        ],
        "medicamentos": [
            {"nome": "Prednisolona", "dose": "5mg", "frequencia": "diaria", "ate": "2024-09-20"}
        ],
        "observacoes": "Braquicefálico. Evitar exercícios intensos e calor."
    }
]

print(f"{len(PETS)} pets carregados.")
for pet in PETS:
    print(f"- {pet['nome']} ({pet['especie']}, {pet['raca']})")

## 3. Ferramentas do assistente
Funções que o VetBot pode chamar para buscar informações dos pets (Function Calling).

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

def buscar_pet(nome: str) -> dict:
    nome_lower = nome.lower()
    for pet in PETS:
        if pet["nome"].lower() == nome_lower:
            return {"sucesso": True, "pet": pet}
    return {"sucesso": False, "erro": f"Pet '{nome}' não encontrado."}


def listar_vacinas(nome: str) -> dict:
    resultado = buscar_pet(nome)
    if not resultado["sucesso"]:
        return resultado
    pet = resultado["pet"]
    return {
        "sucesso": True,
        "pet": pet["nome"],
        "vacinas": pet["vacinas"]
    }


def listar_consultas(nome: str) -> dict:
    resultado = buscar_pet(nome)
    if not resultado["sucesso"]:
        return resultado
    pet = resultado["pet"]
    return {
        "sucesso": True,
        "pet": pet["nome"],
        "consultas": pet["consultas"]
    }


def listar_medicamentos(nome: str) -> dict:
    resultado = buscar_pet(nome)
    if not resultado["sucesso"]:
        return resultado
    pet = resultado["pet"]
    return {
        "sucesso": True,
        "pet": pet["nome"],
        "medicamentos": pet["medicamentos"]
    }


def executar_ferramenta(nome: str, argumentos: dict) -> dict:
    if nome == "buscar_pet":
        return buscar_pet(**argumentos)
    elif nome == "listar_vacinas":
        return listar_vacinas(**argumentos)
    elif nome == "listar_consultas":
        return listar_consultas(**argumentos)
    elif nome == "listar_medicamentos":
        return listar_medicamentos(**argumentos)
    else:
        return {"sucesso": False, "erro": f"Ferramenta '{nome}' não autorizada."}


FERRAMENTAS = [
    {
        "name": "buscar_pet",
        "description": "Busca informações completas de um pet pelo nome.",
        "parameters": {
            "type": "object",
            "properties": {
                "nome": {"type": "string", "description": "Nome do pet"}
            },
            "required": ["nome"]
        }
    },
    {
        "name": "listar_vacinas",
        "description": "Lista as vacinas de um pet e as datas de próxima aplicação.",
        "parameters": {
            "type": "object",
            "properties": {
                "nome": {"type": "string", "description": "Nome do pet"}
            },
            "required": ["nome"]
        }
    },
    {
        "name": "listar_consultas",
        "description": "Lista o histórico de consultas veterinárias de um pet.",
        "parameters": {
            "type": "object",
            "properties": {
                "nome": {"type": "string", "description": "Nome do pet"}
            },
            "required": ["nome"]
        }
    },
    {
        "name": "listar_medicamentos",
        "description": "Lista os medicamentos em uso de um pet.",
        "parameters": {
            "type": "object",
            "properties": {
                "nome": {"type": "string", "description": "Nome do pet"}
            },
            "required": ["nome"]
        }
    }
]

print("Ferramentas definidas:")
for f in FERRAMENTAS:
    print(f"- {f['name']}: {f['description']}")

## 4. System Prompt
Instruções que definem o comportamento e as regras do VetBot.

In [ ]:
SYSTEM_PROMPT = """
Você é o VetBot, assistente virtual do LifePet — um app de saúde contínua para pets.

Seu papel é ajudar tutores a acompanhar a saúde dos seus animais de forma preventiva e personalizada.

Você tem acesso a ferramentas que consultam o histórico real de cada pet, incluindo vacinas, consultas e medicamentos.

Regras:
- responda sempre em português;
- seja objetivo, empático e claro;
- use as ferramentas disponíveis para buscar informações antes de responder;
- nunca invente dados sobre vacinas, consultas ou medicamentos;
- se não encontrar o pet, informe educadamente;
- quando identificar algo que exige atenção veterinária, recomende uma consulta;
- responda em até 150 palavras por mensagem;
- não forneça diagnósticos médicos definitivos.

Contexto do sistema:
- os pets cadastrados têm histórico de vacinas, consultas e medicamentos no app;
- o tutor pode perguntar sobre qualquer pet cadastrado pelo nome;
- seu objetivo principal é promover o cuidado preventivo e contínuo do pet.
"""

print("System prompt definido.")

## 5. Base de conhecimento veterinária
Documentos usados pelo sistema RAG para enriquecer as respostas com informações técnicas.

In [ ]:
import numpy as np

DOCUMENTOS = [
    {
        "fonte": "guia-vacinas-caes.md",
        "titulo": "Vacinas essenciais para cães",
        "conteudo": (
            "Cães devem receber a vacina V10 ou V8 anualmente, cobrindo doenças como "
            "cinomose, parvovirose e leptospirose. A vacina antirrábica também é obrigatória "
            "e deve ser aplicada anualmente. Filhotes iniciam o esquema vacinal entre 6 e 8 semanas."
        )
    },
    {
        "fonte": "guia-vacinas-gatos.md",
        "titulo": "Vacinas essenciais para gatos",
        "conteudo": (
            "Gatos devem receber a vacina V4 anualmente, que protege contra rinotraqueíte, "
            "calicivirose, panleucopenia e clamidiose. A vacina antirrábica é recomendada "
            "especialmente para gatos com acesso à rua."
        )
    },
    {
        "fonte": "guia-alimentacao.md",
        "titulo": "Alimentação saudável para pets",
        "conteudo": (
            "Cães e gatos devem receber alimentação balanceada adequada à espécie, idade e porte. "
            "Evitar alimentos como chocolate, uva, cebola e alho, que são tóxicos para pets. "
            "Água fresca deve estar sempre disponível."
        )
    },
    {
        "fonte": "guia-braquicefalicos.md",
        "titulo": "Cuidados com raças braquicefálicas",
        "conteudo": (
            "Raças como Bulldog Francês, Pug e Shih Tzu possuem focinho achatado e podem ter "
            "dificuldades respiratórias. Devem evitar exercícios intensos, calor excessivo e "
            "ambientes com pouca ventilação. Consultas regulares são essenciais."
        )
    },
    {
        "fonte": "guia-checkup.md",
        "titulo": "Frequência de check-ups veterinários",
        "conteudo": (
            "Pets adultos devem fazer check-up veterinário ao menos uma vez por ano. "
            "Pets idosos (acima de 7 anos) devem consultar o veterinário a cada 6 meses. "
            "Filhotes precisam de acompanhamento mais frequente no primeiro ano de vida."
        )
    },
    {
        "fonte": "guia-sinais-alerta.md",
        "titulo": "Sinais de alerta que exigem consulta imediata",
        "conteudo": (
            "Procure um veterinário imediatamente se o pet apresentar: vômitos ou diarreia "
            "persistentes, dificuldade para respirar, perda de apetite por mais de 24 horas, "
            "letargia intensa, convulsões ou sangramentos."
        )
    },
    {
        "fonte": "guia-peso.md",
        "titulo": "Controle de peso em pets",
        "conteudo": (
            "Obesidade em pets aumenta o risco de diabetes, problemas articulares e cardíacos. "
            "O peso ideal varia por raça e porte. O veterinário deve avaliar o escore corporal "
            "do animal em cada consulta e orientar sobre dieta e exercícios."
        )
    },
    {
        "fonte": "guia-medicamentos.md",
        "titulo": "Cuidados com medicamentos para pets",
        "conteudo": (
            "Nunca administre medicamentos humanos a pets sem orientação veterinária. "
            "Remédios como paracetamol e ibuprofeno são tóxicos para cães e gatos. "
            "Siga sempre a dose e a frequência prescritas pelo veterinário."
        )
    }
]

print(f"{len(DOCUMENTOS)} documentos carregados na base de conhecimento.")
for doc in DOCUMENTOS:
    print(f"- {doc['fonte']}: {doc['titulo']}")

## 6. Embeddings
Transformação dos documentos em vetores numéricos para permitir busca por similaridade semântica.

In [ ]:
EMBEDDING_MODEL = "gemini-embedding-001"

def gerar_embedding(texto: str) -> list:
    resposta = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texto
    )
    return resposta.embeddings[0].values


def similaridade(vetor_a: list, vetor_b: list) -> float:
    a = np.array(vetor_a)
    b = np.array(vetor_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Gerando embeddings dos documentos...")

for doc in DOCUMENTOS:
    doc["embedding"] = gerar_embedding(doc["conteudo"])

print("Embeddings gerados com sucesso.")

## 7. Busca RAG
Função que encontra os documentos mais relevantes para cada pergunta do tutor.

In [ ]:
def buscar_documentos(pergunta: str, top_k: int = 3) -> list:
    embedding_pergunta = gerar_embedding(pergunta)

    pontuados = []
    for doc in DOCUMENTOS:
        score = similaridade(embedding_pergunta, doc["embedding"])
        pontuados.append((score, doc))

    pontuados.sort(key=lambda x: x[0], reverse=True)

    return [doc for _, doc in pontuados[:top_k]]


def formatar_contexto(documentos: list) -> str:
    contexto = ""
    for doc in documentos:
        contexto += f"## {doc['titulo']}\n{doc['conteudo']}\n\n"
    return contexto.strip()


print("Funções de busca RAG definidas.")

## 8. Assistente conversacional
Função principal que combina RAG, Function Calling e histórico de conversa para gerar respostas personalizadas.

In [ ]:
from google import genai
from google.genai import types
import json

def criar_ferramentas():
    return [
        {
            "type": "function",
            "name": "buscar_pet",
            "description": "Busca informações completas de um pet pelo nome.",
            "parameters": {
                "type": "object",
                "properties": {
                    "nome": {"type": "string", "description": "Nome do pet."}
                },
                "required": ["nome"]
            }
        },
        {
            "type": "function",
            "name": "listar_vacinas",
            "description": "Lista as vacinas de um pet e as datas de próxima aplicação.",
            "parameters": {
                "type": "object",
                "properties": {
                    "nome": {"type": "string", "description": "Nome do pet."}
                },
                "required": ["nome"]
            }
        },
        {
            "type": "function",
            "name": "listar_consultas",
            "description": "Lista o histórico de consultas veterinárias de um pet.",
            "parameters": {
                "type": "object",
                "properties": {
                    "nome": {"type": "string", "description": "Nome do pet."}
                },
                "required": ["nome"]
            }
        },
        {
            "type": "function",
            "name": "listar_medicamentos",
            "description": "Lista os medicamentos em uso de um pet.",
            "parameters": {
                "type": "object",
                "properties": {
                    "nome": {"type": "string", "description": "Nome do pet."}
                },
                "required": ["nome"]
            }
        }
    ]

FUNCOES_AUTORIZADAS = {
    "buscar_pet",
    "listar_vacinas",
    "listar_consultas",
    "listar_medicamentos"
}

def conversar(pergunta, historico):
    documentos = buscar_documentos(pergunta, top_k=3)
    contexto = formatar_contexto(documentos)

    system = SYSTEM_PROMPT + f"\n\nCONTEXTO DA BASE DE CONHECIMENTO:\n{contexto}"

    parametros = {
        "model": MODEL,
        "input": pergunta,
        "tools": criar_ferramentas(),
        "system_instruction": system
    }

    if historico:
        parametros["previous_interaction_id"] = historico[-1]["interaction_id"]

    interaction = client.interactions.create(**parametros)

    function_calls = [
        step for step in interaction.steps
        if step.type == "function_call"
    ]

    if function_calls:
        resultados = []
        for call in function_calls:
            nome_funcao = call.name
            argumentos = call.arguments
            if nome_funcao in FUNCOES_AUTORIZADAS:
                resultado = executar_ferramenta(nome_funcao, argumentos)
            else:
                resultado = {"erro": "Ferramenta não autorizada."}

            resultados.append({
                "type": "function_result",
                "name": nome_funcao,
                "call_id": call.id,
                "result": [{"type": "text", "text": json.dumps(resultado, ensure_ascii=False)}]
            })

        interaction = client.interactions.create(
            model=MODEL,
            previous_interaction_id=interaction.id,
            input=resultados,
            tools=criar_ferramentas(),
            system_instruction=system
        )

    texto = interaction.output_text or "Desculpe, não consegui processar sua pergunta."

    historico.append({
        "pergunta": pergunta,
        "resposta": texto,
        "interaction_id": interaction.id
    })

    return texto, historico

print("Assistente conversacional definido.")

## 9. Teste automatizado
Validação do assistente com perguntas sobre diferentes pets.

In [ ]:
import time

historico = []

perguntas = [
    "Quais são as vacinas do Thor e quando vencem?",
    "E quais são as vacinas do Bob?"
]

for pergunta in perguntas:
    print(f"Tutor: {pergunta}")
    resposta, historico = conversar(pergunta, historico)
    print(f"VetBot: {resposta}")
    print("-" * 60)
    time.sleep(5)

## 10. Chat interativo
Interface de conversa em tempo real com o VetBot.

In [ ]:
import time

print("VetBot iniciado! Digite 'sair' para encerrar.\n")

historico = []

while True:
    pergunta = input("Tutor: ")

    if pergunta.lower() == "sair":
        print("VetBot: Até logo! Cuide bem do seu pet.")
        break

    if not pergunta.strip():
        continue

    resposta, historico = conversar(pergunta, historico)
    print(f"VetBot: {resposta}\n")
    time.sleep(3)